# Análisis de interpretabilidad de los modelos — TFG detección aérea

Notebook de análisis para el capítulo de *interpretabilidad / análisis del modelo*.
Corre sobre los modelos **float `.pt`** (los métodos de gradiente necesitan backprop; el grafo INT8
no lo permite). El TFLite INT8 sólo se usa en §14 para medir el coste de la cuantización.

**Cambios de esta revisión (léelo):**
1. Rutas centralizadas arriba y en mayúscula `TFG`; **celda de chequeo** que avisa en ROJO de lo que falta
   (antes las celdas se saltaban en silencio y el notebook parecía correr sin producir nada).
2. `%pip` sin `2>/dev/null` (rompía la instalación).
3. Validación con *fallback* de split (`test`→`val` si no existe) para no reventar.
4. Validación a **320 (despliegue) y 640 (titular de entrenamiento)** para que las cifras no confundan.
5. t-SNE nuevo: **los mismos crops bird/drone por ambos modelos** (comparación limpia especialista vs unified).
6. Grad-CAM/SmoothGrad: parámetros congelados (más rápido) y grad garantizado por la entrada.

---
**Revisión tras reunión con el tutor (añadidos):**
7. Cada figura de interpretabilidad se genera ahora para **los dos modelos** (especialista y unificado), no solo el especialista.
8. §8b **saliency *vanilla*** (gradiente crudo) además de SmoothGrad, para ambos modelos.
9. §12b **curvas ROC *one-vs-rest* + AUC** por clase (a nivel de clasificación; se aclara por qué en detección la métrica es PR/mAP).
10. §12c prueba de **falsos positivos sobre fondo puro** (opcional) + matriz de confusión de **conteos** (fila *background* visible).
11. §6 barrido de distancia con recortes **cuadrados** (sin letterbox: las bandas negras diluían el CAM) + variante letterbox como comparación.
12. §14b demo de **brecha de dominio** (dataset crudo → realce + letterbox del móvil).
13. Clave de Roboflow por `getpass` (nunca en texto plano).


## 1 · Configuración de rutas

Rellena/ajusta estas rutas. Todo lo demás se deriva de aquí. Deja en `None` lo que no tengas.
**OJO Drive es case-sensitive:** usa `TFG` (mayúscula) tal y como está en tu Drive.

In [ ]:
from pathlib import Path
import os

# ============================ RUTAS DEL PROYECTO ============================
# Modelos float .pt (los que usa el análisis: necesita backprop, no vale TFLite)
MODELS = {
    "dronebird": "/content/drive/MyDrive/TFG/models/tfg_drone_bird/weights/best.pt",
    "unified":   "/content/drive/MyDrive/TFG/models/unified_aod4/run/weights/best.pt",
    "heliplane": None,
}

# data.yaml por modelo. Se SOBRESCRIBEN en §2b con lo que descargue Roboflow.
DATA_YAML = {
    "dronebird": "/content/drive/MyDrive/TFG/data/dronebird/data.yaml",
    "unified":   "/content/Airborne-Object-Detection-4-AOD4-4/data.yaml",
    "heliplane": None,
}

# Imágenes + labels YOLO. También se SOBRESCRIBEN en §2b.
YOLO_DATA = {
    "dronebird": {"images": "/content/drive/MyDrive/TFG/data/dronebird/valid/images",
                  "labels": "/content/drive/MyDrive/TFG/data/dronebird/valid/labels"},
    "unified":   {"images": "/content/Airborne-Object-Detection-4-AOD4-4/valid/images",
                  "labels": "/content/Airborne-Object-Detection-4-AOD4-4/valid/labels"},
    "heliplane": None,
}

# INT8 TFLite (solo §14). OJO: el "unified_int8" es en realidad FP16 (ver nota abajo).
TFLITE = {
    "dronebird": "/content/drive/MyDrive/TFG/models/tfg_drone_bird/weights/best_int8.tflite",
    "unified":   "/content/drive/MyDrive/TFG/models/unified/unified_int8.tflite",
    "heliplane": None,
}

# Carpetas de crops para el t-SNE (las GENERA §11 desde YOLO_DATA; aún no existen).
CROPS = {
    "dronebird": "/content/drive/MyDrive/TFG/analysis/crops_dronebird",
    "unified":   "/content/drive/MyDrive/TFG/analysis/crops_unified",
}
CROPS_SHARED = "/content/drive/MyDrive/TFG/analysis/crops_shared_birddrone"

# Barrido de distancia y muestra (ya subidos por ti; confirmados)
DISTANCE_DIR = "/content/drive/MyDrive/TFG/analysis/distance_sweep"
SAMPLE_IMG   = "/content/drive/MyDrive/TFG/analysis/sample.jpg"
# (Opcional) Imágenes de SOLO fondo (cielo limpio, sin objetos) para la prueba de falsos
# positivos (§12c). Déjalo en None si no tienes; esa celda se salta sola y NO crea la carpeta.
BACKGROUND_DIR = None   # p.ej. "/content/drive/MyDrive/TFG/analysis/background"
# ===========================================================================

# Tamaño de análisis: 320 (Acquisition) o 512 (Confirmation)
IMGSZ = 320
IMGSZ_VAL = [320, 640]     # validación a dos tamaños
VAL_SPLIT = "test"         # con fallback a 'val' si no existe

# --------- SALVAGUARDA: no crear NADA si el TFG real no está montado ---------
TFG_ROOT = Path("/content/drive/MyDrive/TFG")
OUT_DIR  = TFG_ROOT / "analysis" / "figuras"

if not (TFG_ROOT / "models").is_dir():
    raise RuntimeError(
        "No veo /MyDrive/TFG/models. Drive NO está montado en el proyecto real.\n"
        "NO se ha creado ninguna carpeta. Monta Drive correctamente (§2) y re-ejecuta esta celda.\n"
        "Comprueba con: os.listdir('/content/drive/MyDrive/TFG')  ->  debe incluir 'models'."
    )

# Llegados aquí, estamos dentro del TFG real: crear la carpeta de figuras es seguro.
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("[OK] Proyecto real detectado. Salida de figuras:", OUT_DIR)

## 1b · Chequeo de rutas — *que el silencio no te engañe*

Antes de nada, comprueba qué existe. Si algo sale como **[FALTA]**, el análisis que dependa de ello se
saltará; arréglalo aquí y no más abajo.

In [ ]:
import os
def _check(label, path):
    ok = bool(path) and os.path.exists(path)
    mark = "[ OK ]" if ok else ("[FALTA]" if path else "[None]")
    print(f"{mark}  {label:26s} {path}")

print("— Modelos .pt —")
for k,v in MODELS.items():   _check(f"MODELS[{k}]", v)
print("— data.yaml —")
for k,v in DATA_YAML.items():_check(f"DATA_YAML[{k}]", v)
print("— YOLO images/labels —")
for k,v in (YOLO_DATA or {}).items():
    if v: _check(f"YOLO[{k}].images", v["images"]); _check(f"YOLO[{k}].labels", v["labels"])
print("— TFLite INT8 —")
for k,v in TFLITE.items():   _check(f"TFLITE[{k}]", v)
print("— Otros —")
_check("DISTANCE_DIR", DISTANCE_DIR); _check("SAMPLE_IMG", SAMPLE_IMG)
_check("BACKGROUND_DIR", BACKGROUND_DIR)
for k,v in CROPS.items():    _check(f"CROPS[{k}]", v)
_check("CROPS_SHARED", CROPS_SHARED)

## 2 · Instalación e imports

In [ ]:
# En Colab. Si ya tienes ultralytics, es casi instantáneo. (Sin 2>/dev/null: rompía el %pip.)
%pip install -q ultralytics scikit-learn pyyaml
# UMAP es opcional (t-SNE ya viene con sklearn). Descomenta si lo quieres:
# %pip install -q umap-learn

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Drive ya montado o entorno no-Colab:", e)

import os, math, glob, yaml
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__, "| Device:", DEVICE)
plt.rcParams["figure.dpi"] = 120

## 2b · Descarga de datasets (Roboflow)

Descarga los dos datasets y **cablea** `DATA_YAML` y `YOLO_DATA` a lo descargado, sobrescribiendo las
rutas de §1. La API key se pide por pantalla con `getpass` (no queda en texto plano).

Nota: los datasets caen en `/content`, que es **efímero**; re-ejecuta esta celda tras cada reinicio del
entorno. Las figuras sí se guardan en Drive (`OUT_DIR`).


In [ ]:
# =============== DESCARGA DE DATASETS (Roboflow) ===============
%pip install -q roboflow
from roboflow import Roboflow
from getpass import getpass
import os, yaml

# Clave de Roboflow: NUNCA hardcodeada. Variable de entorno o getpass (no queda en el .ipynb).
_RF_KEY = os.environ.get("ROBOFLOW_API_KEY") or getpass("Roboflow API key: ")
rf = Roboflow(api_key=_RF_KEY)

# --- Versiones (ajusta si hace falta) ---
AOD4_VERSION      = 4      # AOD4: versión 4 (la que entrenaste)
DRONEBIRD_VERSION = None   # None => detecta automáticamente la última versión

def _download(workspace, project_id, version, fmt="yolov8"):
    proj = rf.workspace(workspace).project(project_id)
    if version is None:
        try:
            def _vnum(v):
                raw = getattr(v, "version", v)
                return int(str(raw).rstrip("/").split("/")[-1])
            version = max(_vnum(v) for v in proj.versions())
            print(f"  [auto] última versión detectada: {version}")
        except Exception as e:
            raise RuntimeError(
                f"No pude autodetectar la versión de {project_id} ({e}). "
                f"Abre el dataset en Roboflow, mira el número de versión y fíjalo a mano.")
    return proj.version(version).download(fmt)

print("Descargando AOD4 (unificado)…")
ds_unified   = _download("airborne-object-detection", "airborne-object-detection-4-aod4", AOD4_VERSION)
print("  ->", ds_unified.location)

print("Descargando Drone-vs-Birds (especialista)…")
ds_dronebird = _download("eliesaade-abmjh", "drone-vs-birds-dvaqh", DRONEBIRD_VERSION)
print("  ->", ds_dronebird.location)

# --- Cablear rutas a lo descargado (sobrescribe las de §1) ---
DATA_YAML["unified"]   = os.path.join(ds_unified.location,   "data.yaml")
DATA_YAML["dronebird"] = os.path.join(ds_dronebird.location, "data.yaml")
YOLO_DATA["unified"]   = {"images": os.path.join(ds_unified.location,   "valid/images"),
                          "labels": os.path.join(ds_unified.location,   "valid/labels")}
YOLO_DATA["dronebird"] = {"images": os.path.join(ds_dronebird.location, "valid/images"),
                          "labels": os.path.join(ds_dronebird.location, "valid/labels")}

# --- Verificación rápida ---
for k in ("dronebird", "unified"):
    p = DATA_YAML[k]; ok = os.path.exists(p)
    print(f"[{'OK' if ok else 'FALTA'}] DATA_YAML[{k}] -> {p}")
    if ok:
        d = yaml.safe_load(open(p))
        print("      names:", d.get("names"),
              "| splits:", [s for s in ("train","val","test") if s in d and d[s]])
        for sub in ("images","labels"):
            q = YOLO_DATA[k][sub]
            print(f"      YOLO_DATA[{k}][{sub}]: {'OK ' if os.path.isdir(q) else 'FALTA'} {q}")
print("\nListo. Puedes re-ejecutar §1b para confirmar [OK] en todo.")


## 3 · Carga de modelos y árbol de módulos

Aquí **verificamos** los índices de las tres cabezas de detección en lugar de asumirlos. En YOLOv8 la
cabeza `Detect` toma como entrada las capas `[15, 18, 21]` (bloques `C2f`):

- **capa 15 → P3 (stride 8)** — objetos *pequeños* → aves y drones a distancia.
- **capa 18 → P4 (stride 16)** — objetos medianos.
- **capa 21 → P5 (stride 32)** — objetos *grandes* → aeronaves.

También localizamos el índice de **SPPF** (para el embedding del t-SNE) leyéndolo de la red, no a mano.
Congelamos los parámetros: los CAM/saliency no necesitan gradiente en los pesos (lo garantizamos por la
entrada), y así el backprop es más rápido.

In [ ]:
models = {}
for key, path in MODELS.items():
    if path and Path(path).exists():
        m = YOLO(path)
        m.model.to(DEVICE).eval()
        for p in m.model.parameters():   # congela pesos: grad sólo por la entrada/activaciones
            p.requires_grad_(False)
        models[key] = m
        print(f"[OK] {key:10s} clases={m.names}")
    else:
        print(f"[--] {key:10s} sin fichero, se omite")

assert models, "No se cargó ningún modelo. Revisa MODELS y el chequeo de rutas (§1b)."

# Inspecciona la red de un modelo de referencia
ref = next(iter(models.values()))
dm  = ref.model            # DetectionModel (nn.Module)
seq = dm.model             # nn.Sequential de capas
print("\nÍndice | módulo | from")
for i, layer in enumerate(seq):
    print(f"{i:6d} | {layer.__class__.__name__:14s} | from={getattr(layer,'f','')}")

# Índices P3/P4/P5 desde Detect.f (fuente de verdad)
detect = seq[-1]
print("\nDetect.f (entradas P3/P4/P5) =", detect.f)
TARGET_LAYERS = {"P3": 15, "P4": 18, "P5": 21}
if hasattr(detect, "f") and isinstance(detect.f, (list, tuple)) and len(detect.f) == 3:
    TARGET_LAYERS = {"P3": detect.f[0], "P4": detect.f[1], "P5": detect.f[2]}
print("TARGET_LAYERS =", TARGET_LAYERS)

# Índice de SPPF (para el embedding del backbone), localizado por tipo de módulo
SPPF_IDX = None
for i, layer in enumerate(seq):
    if layer.__class__.__name__.upper() == "SPPF":
        SPPF_IDX = i
if SPPF_IDX is None:
    SPPF_IDX = 9   # por defecto en YOLOv8
    print("No se encontró SPPF; usando índice 9 por defecto.")
print("SPPF_IDX =", SPPF_IDX)

## 4 · Utilidades — letterbox, preprocesado y overlay

Replicamos el letterbox de despliegue para que la rejilla del mapa de activación coincida con la imagen.

In [ ]:
def letterbox(im_bgr, new=None, color=(114,114,114)):
    """Redimensiona con padding manteniendo aspect ratio (como Ultralytics)."""
    new = new or IMGSZ
    h, w = im_bgr.shape[:2]
    r = min(new / h, new / w)
    nh, nw = int(round(h*r)), int(round(w*r))
    resized = cv2.resize(im_bgr, (nw, nh), interpolation=cv2.INTER_LINEAR)
    top = (new-nh)//2; bottom = new-nh-top
    left = (new-nw)//2; right = new-nw-left
    return cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)

def preprocess(path_or_bgr, imgsz=None):
    """Devuelve (tensor [1,3,H,W] en 0-1, imagen letterboxed BGR para overlay)."""
    imgsz = imgsz or IMGSZ
    im = cv2.imread(path_or_bgr) if isinstance(path_or_bgr, str) else path_or_bgr
    if im is None:
        raise FileNotFoundError(f"No se pudo leer la imagen: {path_or_bgr}")
    lb = letterbox(im, imgsz)
    rgb = lb[:, :, ::-1].astype(np.float32) / 255.0
    t = torch.from_numpy(np.ascontiguousarray(rgb.transpose(2, 0, 1)))[None].to(DEVICE)
    return t, lb

def overlay_cam(lb_bgr, cam, alpha=0.5):
    """Superpone un mapa CAM [h,w] en 0-1 sobre la imagen letterboxed BGR. Devuelve RGB."""
    cam_r = cv2.resize(cam.astype(np.float32), (lb_bgr.shape[1], lb_bgr.shape[0]))
    heat = cv2.applyColorMap(np.uint8(255*cam_r), cv2.COLORMAP_JET)
    out = cv2.addWeighted(lb_bgr, 1-alpha, heat, alpha, 0)
    return out[:, :, ::-1]

def norm01(x):
    x = x - x.min()
    return x / (x.max() + 1e-8)

def clean_ax(ax, keep_labels=False):
    """Oculta ticks y marco pero permite conservar xlabel/ylabel (para etiquetas de fila)."""
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)
    if not keep_labels:
        ax.set_xlabel(""); ax.set_ylabel("")

def preprocess_square(path_or_bgr, imgsz=None):
    """Como preprocess pero SIN letterbox: recorta el cuadrado central y reescala a imgsz.
    Evita las bandas negras del letterbox (que diluyen los mapas CAM). Es una elección de
    VISUALIZACIÓN: el pipeline de despliegue en el móvil SÍ hace letterbox (ver §14b)."""
    imgsz = imgsz or IMGSZ
    im = cv2.imread(path_or_bgr) if isinstance(path_or_bgr, str) else path_or_bgr
    if im is None:
        raise FileNotFoundError(f"No se pudo leer la imagen: {path_or_bgr}")
    h, w = im.shape[:2]
    s = min(h, w)
    top = (h - s) // 2; left = (w - s) // 2
    sq = im[top:top+s, left:left+s]
    sq = cv2.resize(sq, (imgsz, imgsz), interpolation=cv2.INTER_LINEAR)
    rgb = sq[:, :, ::-1].astype(np.float32) / 255.0
    t = torch.from_numpy(np.ascontiguousarray(rgb.transpose(2, 0, 1)))[None].to(DEVICE)
    return t, sq

def _prep(path_or_bgr, imgsz=None, square=False):
    """Selector: letterbox (despliegue real) o cuadrado central (visualización nítida)."""
    return preprocess_square(path_or_bgr, imgsz) if square else preprocess(path_or_bgr, imgsz)

def analyze_models():
    """Itera (key, model) sobre los modelos cargados que queremos analizar.
    Centraliza el 'haz cada figura para los dos modelos' que pidió el tutor."""
    for k in ("dronebird", "unified"):
        if k in models:
            yield k, models[k]

_TITLES = {"dronebird": "Especialista Drone/Bird", "unified": "Unificado AOD4"}
def _suffix(key): return "" if key == "dronebird" else f"_{key}"


## 5 · EigenCAM multi-escala — *¿dónde mira el modelo?*

EigenCAM es **libre de gradiente**: proyecta las activaciones de una capa sobre su primera componente
principal. Es lo estándar y robusto para YOLOv8 (evita el "¿gradiente de qué detección?" de los detectores
densos). Lo hacemos a mano con *hooks* (más transparente para explicarlo en la memoria).

Generamos el CAM en **P3/P4/P5**. Para aves y drones a distancia, el mapa **P3** es el informativo.

In [ ]:
def _hook_activation(dm, idx, store):
    layer = dm.model[idx]
    def fn(m, i, o): store["act"] = o.detach()
    return layer.register_forward_hook(fn)

def eigencam(model, img, idx, imgsz=None, square=False):
    """CAM de EigenCAM en la capa idx. Devuelve (cam[h,w] 0-1, lb_bgr)."""
    imgsz = imgsz or IMGSZ
    dm = model.model
    t, lb = _prep(img, imgsz, square)
    store = {}
    h = _hook_activation(dm, idx, store)
    with torch.no_grad():
        _ = dm(t)
    h.remove()
    A = store["act"][0]                     # [C,h,w]
    C, hh, ww = A.shape
    P = A.permute(1, 2, 0).reshape(-1, C)   # [h*w, C]
    P = P - P.mean(0, keepdim=True)
    U, S, Vh = torch.linalg.svd(P, full_matrices=False)
    proj = (P @ Vh[0]).reshape(hh, ww)      # 1a componente principal
    cam = norm01(proj.abs()).cpu().numpy()  # abs() resuelve la ambiguedad de signo del PCA
    return cam, lb

def show_eigencam_multiscale(model, img, title="", save_as=None, square=False):
    fig, ax = plt.subplots(1, 4, figsize=(16, 4))
    _, lb = _prep(img, None, square)
    ax[0].imshow(lb[:, :, ::-1]); ax[0].set_title("Entrada"); clean_ax(ax[0])
    for j, (name, idx) in enumerate(TARGET_LAYERS.items(), start=1):
        cam, lb = eigencam(model, img, idx, square=square)
        ax[j].imshow(overlay_cam(lb, cam)); ax[j].set_title(f"EigenCAM {name} (capa {idx})"); clean_ax(ax[j])
    fig.suptitle(title, fontsize=13); plt.tight_layout()
    if save_as: fig.savefig(save_as, bbox_inches="tight"); print("Guardado:", save_as)
    plt.show()

# Para los DOS modelos (el tutor pidio no limitarse al especialista).
if Path(SAMPLE_IMG).exists() and any(True for _ in analyze_models()):
    for key, m in analyze_models():
        show_eigencam_multiscale(m, SAMPLE_IMG,
                                 f"{_TITLES.get(key,key)} — EigenCAM multi-escala",
                                 save_as=str(OUT_DIR / f"eigencam_multiescala{_suffix(key)}.png"))
else:
    print("Configura SAMPLE_IMG y al menos un modelo para ver EigenCAM.")


## 6 · Barrido de distancia — degradación de la atención *(figura clave)*

Conecta la interpretabilidad con el experimento de campo. Sobre el mismo objeto a distancias crecientes, la
atención P3 debería **concentrarse** de cerca y **dispersarse** al caer los pixels-on-target. Evidencia
cualitativa que acompaña al barrido cuantitativo de 5 m.

Nombra las imágenes para que ordenen: `05m.jpg`, `10m.jpg`, …

In [ ]:
def distance_contact_sheet(model, folder, layer_name="P3", imgsz=None, save_as=None, square=True):
    imgsz = imgsz or IMGSZ
    idx = TARGET_LAYERS[layer_name]
    paths = sorted(glob.glob(os.path.join(folder, "*.jpg")) +
                   glob.glob(os.path.join(folder, "*.png")))
    if not paths:
        print(f"Sin imagenes en {folder} — se omite el barrido."); return
    n = len(paths)
    fig, ax = plt.subplots(2, n, figsize=(3*n, 6.2))
    if n == 1: ax = ax.reshape(2, 1)
    for j, p in enumerate(paths):
        cam, lb = eigencam(model, p, idx, imgsz, square=square)
        ax[0, j].imshow(lb[:, :, ::-1]); clean_ax(ax[0, j]); ax[0, j].set_title(Path(p).stem, fontsize=10)
        ax[1, j].imshow(overlay_cam(lb, cam)); clean_ax(ax[1, j])
    clean_ax(ax[0, 0], keep_labels=True); ax[0, 0].set_ylabel("Entrada", fontsize=11)
    clean_ax(ax[1, 0], keep_labels=True); ax[1, 0].set_ylabel(f"EigenCAM {layer_name}", fontsize=11)
    modo = "cuadrado" if square else "letterbox"
    fig.suptitle(f"Degradacion de la atencion con la distancia — {layer_name} ({modo})", fontsize=13)
    plt.tight_layout()
    if save_as: fig.savefig(save_as, bbox_inches="tight"); print("Guardado:", save_as)
    plt.show()

# El tutor pidio CUADRADO (sin letterbox): las bandas negras diluian el mapa.
# Nombre canonico = version cuadrada (la que usa la memoria); para AMBOS modelos.
for key, m in analyze_models():
    distance_contact_sheet(m, DISTANCE_DIR, "P3", square=True,
                           save_as=str(OUT_DIR / f"barrido_distancia_P3{_suffix(key)}.png"))
# Comparacion honesta letterbox vs cuadrado (especialista) para el apendice:
if "dronebird" in models:
    distance_contact_sheet(models["dronebird"], DISTANCE_DIR, "P3", square=False,
                           save_as=str(OUT_DIR / "barrido_distancia_P3_letterbox.png"))


## 7 · Grad-CAM dirigido — *¿por qué esta clase y no otra?*

A diferencia de EigenCAM, Grad-CAM es **dirigido**: elegimos una clase y vemos qué regiones elevan su
puntuación. Útil para *bird vs drone*. Backprop manual a la capa objetivo (con `retain_grad`); el gradiente
se garantiza haciendo que la **entrada** requiera grad (los pesos están congelados).

In [ ]:
def gradcam(model, img, class_idx, layer_name="P3", imgsz=None, square=False):
    imgsz = imgsz or IMGSZ
    dm = model.model
    idx = TARGET_LAYERS[layer_name]
    t, lb = _prep(img, imgsz, square)
    t = t.detach().clone().requires_grad_(True)   # grad por la entrada (pesos congelados)
    store = {}
    layer = dm.model[idx]
    def fhook(m, i, o):
        o.retain_grad(); store["act"] = o
    h = layer.register_forward_hook(fhook)
    dm.zero_grad(set_to_none=True)
    out = dm(t)
    out = out[0] if isinstance(out, (list, tuple)) else out   # [1, 4+nc, N]
    nc = len(model.names)
    scores = out[0, 4:4+nc, :]          # [nc, N]
    target = scores[class_idx].max()    # pico de esa clase sobre todos los anchors
    target.backward()
    h.remove()
    act  = store["act"][0]              # [C,h,w]
    grad = store["act"].grad[0]         # [C,h,w]
    weights = grad.mean(dim=(1, 2))     # [C]
    cam = torch.relu((weights[:, None, None] * act).sum(0))
    return norm01(cam).detach().cpu().numpy(), lb

def gradcam_all_classes(model, img, title, save_as, square=False):
    names = model.names
    ncols = len(names) + 1
    fig, ax = plt.subplots(1, ncols, figsize=(5*ncols, 4.5))
    if ncols == 1: ax = [ax]
    _, lb = _prep(img, None, square)
    ax[0].imshow(lb[:, :, ::-1]); ax[0].set_title("Entrada"); clean_ax(ax[0])
    for c, cname in names.items():
        cam, lb = gradcam(model, img, c, "P3", square=square)
        ax[c+1].imshow(overlay_cam(lb, cam)); ax[c+1].set_title(f"Grad-CAM: {cname}"); clean_ax(ax[c+1])
    fig.suptitle(title, fontsize=13); plt.tight_layout()
    fig.savefig(save_as, bbox_inches="tight"); print("Guardado:", save_as); plt.show()

if Path(SAMPLE_IMG).exists() and any(True for _ in analyze_models()):
    # Especialista: nombre canonico que ya usa la memoria.
    if "dronebird" in models:
        gradcam_all_classes(models["dronebird"], SAMPLE_IMG,
                            "Especialista Drone/Bird — Grad-CAM por clase",
                            str(OUT_DIR / "gradcam_bird_drone.png"))
    # Unificado: las 4 clases. Clave para VER por que etiqueta el dron como 'airplane'.
    if "unified" in models:
        gradcam_all_classes(models["unified"], SAMPLE_IMG,
                            "Unificado AOD4 — Grad-CAM por clase",
                            str(OUT_DIR / "gradcam_unified.png"))
else:
    print("Configura SAMPLE_IMG y los modelos.")


## 8 · SmoothGrad — saliency a nivel de píxel

Saliency de gradiente de entrada, promediado sobre copias con ruido (~40) para reducir el ruido del
gradiente crudo. Complementa a CAM: píxel a píxel en vez de por regiones.

In [ ]:
def smoothgrad(model, img, class_idx, imgsz=None, n=40, sigma=0.10, square=False):
    imgsz = imgsz or IMGSZ
    dm = model.model
    t0, lb = _prep(img, imgsz, square)
    nc = len(model.names)
    acc = torch.zeros_like(t0)
    for _ in range(n):
        noise = torch.randn_like(t0) * sigma
        t = (t0 + noise).detach().clone().requires_grad_(True)
        dm.zero_grad(set_to_none=True)
        out = dm(t); out = out[0] if isinstance(out, (list, tuple)) else out
        target = out[0, 4:4+nc, :][class_idx].max()
        target.backward()
        acc += t.grad.abs()
    sal = norm01(acc[0].max(0).values).cpu().numpy()   # max sobre canales
    return sal, lb

def _dominant_class(model):
    """Clase 'drone' si existe (el objeto de interes); si no, la primera."""
    for c, nm in model.names.items():
        if nm == "drone": return c
    return next(iter(model.names))

if Path(SAMPLE_IMG).exists() and any(True for _ in analyze_models()):
    for key, m in analyze_models():
        c = _dominant_class(m)
        sal, lb = smoothgrad(m, SAMPLE_IMG, c, n=40)
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        ax[0].imshow(lb[:, :, ::-1]); ax[0].set_title("Entrada"); clean_ax(ax[0])
        ax[1].imshow(sal, cmap="hot"); ax[1].set_title(f"SmoothGrad ({m.names[c]})"); clean_ax(ax[1])
        plt.tight_layout()
        fig.savefig(OUT_DIR / f"smoothgrad{_suffix(key)}.png", bbox_inches="tight")
        print("Guardado:", OUT_DIR / f"smoothgrad{_suffix(key)}.png"); plt.show()
else:
    print("Configura SAMPLE_IMG.")


## 8b · Saliency *vanilla* — gradiente crudo (Simonyan 2013)

Saliency básica: gradiente de la puntuación de la clase respecto a los píxeles de entrada, **sin**
promediar sobre ruido (eso es SmoothGrad, §8). El contraste vanilla↔SmoothGrad es en sí informativo:
SmoothGrad limpia el ruido de *sal y pimienta* del gradiente crudo. Se genera para **ambos** modelos.


In [ ]:
def vanilla_saliency(model, img, class_idx, imgsz=None, square=False):
    imgsz = imgsz or IMGSZ
    dm = model.model
    t0, lb = _prep(img, imgsz, square)
    nc = len(model.names)
    t = t0.detach().clone().requires_grad_(True)
    dm.zero_grad(set_to_none=True)
    out = dm(t); out = out[0] if isinstance(out, (list, tuple)) else out
    target = out[0, 4:4+nc, :][class_idx].max()
    target.backward()
    sal = norm01(t.grad.abs()[0].max(0).values).cpu().numpy()
    return sal, lb

if Path(SAMPLE_IMG).exists() and any(True for _ in analyze_models()):
    for key, m in analyze_models():
        c = _dominant_class(m)
        sal, lb = vanilla_saliency(m, SAMPLE_IMG, c)
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        ax[0].imshow(lb[:, :, ::-1]); ax[0].set_title("Entrada"); clean_ax(ax[0])
        ax[1].imshow(sal, cmap="hot"); ax[1].set_title(f"Saliency vanilla ({m.names[c]})"); clean_ax(ax[1])
        plt.tight_layout()
        fig.savefig(OUT_DIR / f"saliency_vanilla_{key}.png", bbox_inches="tight")
        print("Guardado:", OUT_DIR / f"saliency_vanilla_{key}.png"); plt.show()
else:
    print("Configura SAMPLE_IMG.")


## 9 · Filtros de la primera capa convolucional

Los kernels de la primera conv son de 3 canales → se renderizan como tiles RGB. Muestran que el modelo
aprendió detectores de bordes/color/orientación. Figura clásica y barata.

In [ ]:
def show_first_conv_filters(model, cols=8, save_as=None):
    dm = model.model
    first = None
    for mod in dm.modules():
        if isinstance(mod, torch.nn.Conv2d) and mod.in_channels == 3:
            first = mod; break
    if first is None:
        print("No se encontro conv de entrada de 3 canales."); return
    W = first.weight.data.clone().cpu()          # [out,3,k,k]
    n = W.shape[0]; rows = math.ceil(n/cols)
    fig, ax = plt.subplots(rows, cols, figsize=(cols*1.1, rows*1.1))
    ax = np.array(ax).reshape(rows, cols)
    for i in range(rows*cols):
        r, cc = divmod(i, cols); ax[r, cc].axis("off")
        if i < n:
            f = W[i]; f = (f - f.min())/(f.max()-f.min()+1e-8)
            ax[r, cc].imshow(f.permute(1, 2, 0).numpy())
    fig.suptitle(f"Filtros de la 1a capa conv ({n} kernels de {W.shape[2]}x{W.shape[3]})", fontsize=12)
    plt.tight_layout()
    if save_as: fig.savefig(save_as, bbox_inches="tight"); print("Guardado:", save_as)
    plt.show()

# Para AMBOS modelos. Nota: los filtros de la 1a capa suelen parecerse mucho entre modelos
# (Gabor-like); si salen casi identicos, es un hallazgo en si mismo (bajo nivel = generico).
for key, m in analyze_models():
    show_first_conv_filters(m, save_as=str(OUT_DIR / f"filtros_capa1{_suffix(key)}.png"))


## 10 · Mapas de activación (feature maps) por bloque

Los filtros profundos no son RGB: se pasa una imagen y se muestran las activaciones de un bloque. Capas
tempranas → textura; profundas → estructura tipo objeto.

In [ ]:
def show_feature_maps(model, img, idx, max_ch=16, imgsz=None, save_as=None):
    imgsz = imgsz or IMGSZ
    dm = model.model
    t, lb = preprocess(img, imgsz)
    store = {}
    h = _hook_activation(dm, idx, store)
    with torch.no_grad(): dm(t)
    h.remove()
    A = store["act"][0].cpu().numpy()   # [C,h,w]
    ch = min(max_ch, A.shape[0]); cols = 8; rows = math.ceil(ch/cols)
    fig, ax = plt.subplots(rows, cols, figsize=(cols*1.3, rows*1.3))
    ax = np.array(ax).reshape(rows, cols)
    for i in range(rows*cols):
        r, cc = divmod(i, cols); ax[r, cc].axis("off")
        if i < ch:
            fm = A[i]; fm = (fm-fm.min())/(fm.max()-fm.min()+1e-8)
            ax[r, cc].imshow(fm, cmap="viridis")
    fig.suptitle(f"Feature maps — capa {idx} ({A.shape[0]} canales, mostrando {ch})", fontsize=12)
    plt.tight_layout()
    if save_as: fig.savefig(save_as, bbox_inches="tight"); print("Guardado:", save_as)
    plt.show()

if Path(SAMPLE_IMG).exists() and any(True for _ in analyze_models()):
    for key, m in analyze_models():
        for idx, tag in [(2, "temprana"), (TARGET_LAYERS["P3"], "P3"), (TARGET_LAYERS["P5"], "P5")]:
            show_feature_maps(m, SAMPLE_IMG, idx,
                              save_as=str(OUT_DIR / f"featuremaps_{tag}{_suffix(key)}.png"))


## 11 · Separabilidad de clases — t-SNE *(figura clave)*

La pregunta central del TFG: **¿por qué un especialista podría ganar a un generalista?** Extraemos un
embedding por objeto (global-average-pool de **SPPF**, el rasgo más profundo del backbone), proyectamos con
t-SNE y coloreamos por clase real.

Dos vistas:
1. **Por modelo** con sus propias clases (especialista bird/drone; unified 4 clases).
2. **Comparación limpia (nueva):** los **mismos crops bird/drone** (de AOD4) pasados por **ambos** modelos.
   Si el unified enreda más bird↔drone que el especialista, aquí se ve directamente — es el *porqué* del
   Cap. 6, con la única variable siendo el modelo.

In [ ]:
# --- Helper: construir crops desde labels YOLO ---
def build_crops_from_yolo(images_dir, labels_dir, class_names, out_dir,
                          max_per_class=250, only_classes=None):
    out = Path(out_dir); counts = {}
    for img_path in glob.glob(os.path.join(images_dir, "*")):
        lab = Path(labels_dir) / (Path(img_path).stem + ".txt")
        if not lab.exists(): continue
        im = cv2.imread(img_path)
        if im is None: continue
        H, W = im.shape[:2]
        for line in lab.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5: continue
            cls = int(parts[0]); cx, cy, w, h = map(float, parts[1:5])
            cname = class_names.get(cls)
            if cname is None: continue
            if only_classes and cname not in only_classes: continue
            counts.setdefault(cname, 0)
            if counts[cname] >= max_per_class: continue
            x1 = max(0, int((cx-w/2)*W)); y1 = max(0, int((cy-h/2)*H))
            x2 = min(W, int((cx+w/2)*W)); y2 = min(H, int((cy+h/2)*H))
            if x2-x1 < 8 or y2-y1 < 8: continue
            d = out / cname; d.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(d / f"{Path(img_path).stem}_{counts[cname]}.jpg"), im[y1:y2, x1:x2])
            counts[cname] += 1
    print("Crops generados en", out_dir, "->", counts)

# --- Generar crops desde los datasets descargados (para el t-SNE de abajo) ---
def _has_crops(d):
    return os.path.isdir(d) and len(glob.glob(os.path.join(d, "*", "*.jpg"))) > 0

if "dronebird" in models and YOLO_DATA.get("dronebird") and not _has_crops(CROPS["dronebird"]):
    build_crops_from_yolo(YOLO_DATA["dronebird"]["images"], YOLO_DATA["dronebird"]["labels"],
                          models["dronebird"].names, CROPS["dronebird"])
else:
    print("crops dronebird: ya existen o faltan datos, se omite.")

if "unified" in models and YOLO_DATA.get("unified") and not _has_crops(CROPS["unified"]):
    build_crops_from_yolo(YOLO_DATA["unified"]["images"], YOLO_DATA["unified"]["labels"],
                          models["unified"].names, CROPS["unified"])
else:
    print("crops unified: ya existen o faltan datos, se omite.")

# Crops COMPARTIDOS bird/drone desde AOD4 (para la figura clave del t-SNE):
if "unified" in models and YOLO_DATA.get("unified") and not _has_crops(CROPS_SHARED):
    build_crops_from_yolo(YOLO_DATA["unified"]["images"], YOLO_DATA["unified"]["labels"],
                          models["unified"].names, CROPS_SHARED, only_classes={"bird","drone"})
else:
    print("crops compartidos bird/drone: ya existen o faltan datos, se omite.")


In [ ]:
from sklearn.manifold import TSNE

def gap_embedding(model, crop_bgr, imgsz=None):
    imgsz = imgsz or IMGSZ
    dm = model.model
    t, _ = preprocess(crop_bgr, imgsz)
    store = {}
    h = _hook_activation(dm, SPPF_IDX, store)
    with torch.no_grad(): dm(t)
    h.remove()
    return store["act"][0].mean(dim=(1, 2)).cpu().numpy()   # GAP -> vector

def _load_crops(crops_dir):
    classes = sorted([d for d in os.listdir(crops_dir)
                      if os.path.isdir(os.path.join(crops_dir, d))]) if os.path.isdir(crops_dir) else []
    files, labels = [], []
    for ci, c in enumerate(classes):
        for p in glob.glob(os.path.join(crops_dir, c, "*.jpg")):
            files.append(p); labels.append(ci)
    return classes, files, labels

def _tsne_plot(ax, X, y, classes, title):
    emb = TSNE(n_components=2, perplexity=min(30, max(5, len(X)//4)),
               init="pca", random_state=0).fit_transform(X)
    for ci, c in enumerate(classes):
        s = np.array(y) == ci
        ax.scatter(emb[s, 0], emb[s, 1], s=12, alpha=0.7, label=c)
    ax.legend(); ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

def embed_and_plot(model, crops_dir, title, save_as=None):
    classes, files, y = _load_crops(crops_dir)
    if len(files) < 10:
        print(f"Muy pocos crops en {crops_dir} — se omite."); return
    X = np.array([gap_embedding(model, cv2.imread(p)) for p in files])
    fig, ax = plt.subplots(figsize=(7, 6)); _tsne_plot(ax, X, y, classes, title)
    if save_as: plt.savefig(save_as, bbox_inches="tight"); print("Guardado:", save_as)
    plt.show()

def embed_two_models_same_crops(model_a, name_a, model_b, name_b, crops_dir, save_as=None):
    """Los MISMOS crops por dos modelos, lado a lado. Variable única = el modelo."""
    classes, files, y = _load_crops(crops_dir)
    if len(files) < 10:
        print(f"Muy pocos crops en {crops_dir} — se omite."); return
    imgs = [cv2.imread(p) for p in files]
    Xa = np.array([gap_embedding(model_a, im) for im in imgs])
    Xb = np.array([gap_embedding(model_b, im) for im in imgs])
    fig, ax = plt.subplots(1, 2, figsize=(13, 6))
    _tsne_plot(ax[0], Xa, y, classes, f"{name_a} — SPPF")
    _tsne_plot(ax[1], Xb, y, classes, f"{name_b} — SPPF")
    fig.suptitle("Separabilidad bird/drone con los MISMOS crops (t-SNE)", fontsize=13)
    plt.tight_layout()
    if save_as: fig.savefig(save_as, bbox_inches="tight"); print("Guardado:", save_as)
    plt.show()

# 1) Por modelo (sus propias clases)
if "dronebird" in models:
    embed_and_plot(models["dronebird"], CROPS["dronebird"],
                   "Especialista Drone/Bird — separabilidad (t-SNE, SPPF)",
                   save_as=str(OUT_DIR / "tsne_especialista.png"))
if "unified" in models:
    embed_and_plot(models["unified"], CROPS["unified"],
                   "Unificado AOD4 — separabilidad (t-SNE, SPPF)",
                   save_as=str(OUT_DIR / "tsne_unificado.png"))

# 2) FIGURA CLAVE: mismos crops bird/drone por ambos modelos
if "dronebird" in models and "unified" in models:
    embed_two_models_same_crops(models["dronebird"], "Especialista",
                                models["unified"], "Unificado",
                                CROPS_SHARED,
                                save_as=str(OUT_DIR / "tsne_comparacion_mismos_crops.png"))

## 12 · Matriz de confusión + curvas PR

Ultralytics las genera al validar. Validamos a **320 (despliegue)** y **640 (como se entrenó)** para que las
cifras no confundan: el mAP a 320 será más bajo que tu titular de 640, y eso es esperado, no una regresión.

In [ ]:
from IPython.display import Image as IPyImage, display

def pick_split(data_yaml, preferred=VAL_SPLIT):
    try:
        d = yaml.safe_load(open(data_yaml))
        if preferred in d and d[preferred]: return preferred
    except Exception:
        pass
    return "val"

def validate_and_show(key, imgszs=None):
    imgszs = imgszs or IMGSZ_VAL
    if key not in models or not DATA_YAML.get(key) or not Path(DATA_YAML[key]).exists():
        print(f"[{key}] sin data.yaml — se omite val."); return
    split = pick_split(DATA_YAML[key])
    for sz in imgszs:
        print(f"\n=== Validando {key} @ imgsz={sz} (split={split}) ===")
        metrics = models[key].val(data=DATA_YAML[key], imgsz=sz, split=split, verbose=False)
        print(f"mAP50={metrics.box.map50:.3f}  mAP50-95={metrics.box.map:.3f}")
        # mAP por clase (rellena tab:map_clase de la memoria)
        try:
            print("        mAP@50 por clase:")
            for ci, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
                print(f"          {models[key].names[int(ci)]:12s} {float(ap):.3f}")
        except Exception as e:
            print("        (no se pudo desglosar por clase:", e, ")")
        save_dir = Path(metrics.save_dir)
        import shutil
        wanted = {"confusion_matrix_normalized.png": f"confusion_{key}_{sz}.png",
                  "confusion_matrix.png": f"confusion_{key}_{sz}_counts.png",  # conteos: fila 'background' visible
                  "PR_curve.png": f"pr_{key}_{sz}.png"}
        for fname, clean in wanted.items():
            fp = save_dir / fname
            if fp.exists():
                shutil.copy(fp, OUT_DIR / clean)          # copia a figuras/ con nombre limpio
                print(f"{clean} (imgsz={sz})"); display(IPyImage(str(OUT_DIR / clean), width=520))

for k in ("dronebird", "unified"):
    validate_and_show(k)

## 12b · Curvas ROC *one-vs-rest* + AUC (a nivel de clasificación)

Aclaración importante (dila en la defensa): **ROC/AUC no es la métrica natural de un *detector*** —necesita
verdaderos negativos bien definidos, y en detección el número de ventanas negativas es indefinido; por eso
YOLO reporta PR/mAP (§12). Aquí ROC se usa a **nivel de clasificación**: dado un recorte de una clase, ¿con
qué fuerza puntúa el modelo esa clase frente al resto? Los negativos son las instancias de las OTRAS clases
→ TN bien definidos → ROC *one-vs-rest* + AUC por clase, legítimo. Para **ambos** modelos.


In [ ]:
from sklearn.metrics import roc_curve, auc

def _class_scores_for_crop(model, crop_bgr, imgsz=None):
    """Vector [nc] de score por clase para un recorte: max confianza de caja de esa clase."""
    imgsz = imgsz or IMGSZ
    nc = len(model.names)
    res = model.predict(crop_bgr, imgsz=imgsz, conf=0.001, verbose=False)[0]
    s = np.zeros(nc, dtype=np.float32)
    if res.boxes is not None and len(res.boxes):
        cls = res.boxes.cls.cpu().numpy().astype(int)
        cf  = res.boxes.conf.cpu().numpy()
        for k, c in zip(cf, cls):
            if 0 <= c < nc and k > s[c]: s[c] = k
    return s

def roc_ovr(key, crops_dir, save_as=None):
    if key not in models: print(f"[{key}] modelo no cargado."); return
    model = models[key]
    classes, files, y = _load_crops(crops_dir)
    if len(files) < 10:
        print(f"[{key}] muy pocos crops en {crops_dir} — se omite ROC."); return
    S = np.array([_class_scores_for_crop(model, cv2.imread(p)) for p in files])
    y = np.array(y)
    fig, ax = plt.subplots(figsize=(6, 6))
    plotted = 0
    for ci, cname in enumerate(classes):
        model_ci = next((c for c, nm in model.names.items() if nm == cname), None)
        if model_ci is None: continue
        y_true = (y == ci).astype(int)
        if y_true.sum() == 0 or y_true.sum() == len(y_true): continue
        fpr, tpr, _ = roc_curve(y_true, S[:, model_ci])
        a = auc(fpr, tpr)
        ax.plot(fpr, tpr, lw=2, label=f"{cname} (AUC={a:.3f})"); plotted += 1
    if plotted == 0:
        print(f"[{key}] no se pudo trazar ninguna clase."); plt.close(fig); return
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set_xlabel("Tasa de falsos positivos"); ax.set_ylabel("Tasa de verdaderos positivos")
    ax.set_title(f"ROC one-vs-rest — {key}"); ax.legend(loc="lower right")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); plt.tight_layout()
    if save_as: fig.savefig(save_as, bbox_inches="tight"); print("Guardado:", save_as)
    plt.show()

roc_ovr("dronebird", CROPS["dronebird"], save_as=str(OUT_DIR / "roc_dronebird.png"))
roc_ovr("unified",   CROPS["unified"],   save_as=str(OUT_DIR / "roc_unified.png"))


## 12c · Prueba de falsos positivos sobre fondo puro (opcional)

El tutor pidió una confusión *completa* con fila **background**. `model.val()` ya la incluye: fíjate en la
fila/columna `background` de la **matriz de conteos** (`confusion_*_counts.png`, copiada en §12). Para
reforzarla con fondo real (cielo limpio sin objetos), pon imágenes en `BACKGROUND_DIR`: aquí medimos
cuántos **falsos positivos** dispara cada modelo sobre ellas. Si `BACKGROUND_DIR` es `None`, la celda se
salta sola (no crea carpetas — respeta la salvaguarda de §1).


In [ ]:
def background_fp_probe(conf=0.25, imgsz=None):
    imgsz = imgsz or IMGSZ
    if not BACKGROUND_DIR or not os.path.isdir(BACKGROUND_DIR):
        print("BACKGROUND_DIR no definido o inexistente — se omite (sin crear nada)."); return
    imgs = sorted(glob.glob(os.path.join(BACKGROUND_DIR, "*.jpg")) +
                  glob.glob(os.path.join(BACKGROUND_DIR, "*.png")))
    if not imgs:
        print(f"Sin imagenes en {BACKGROUND_DIR} — se omite."); return
    stats = {}
    for key, m in analyze_models():
        total_fp = 0; imgs_with_fp = 0
        for p in imgs:
            res = m.predict(p, imgsz=imgsz, conf=conf, verbose=False)[0]
            nb = 0 if res.boxes is None else len(res.boxes)
            total_fp += nb; imgs_with_fp += (nb > 0)
        stats[key] = (total_fp, imgs_with_fp, len(imgs))
        print(f"[{key}] {total_fp} FP en {imgs_with_fp}/{len(imgs)} imagenes de fondo (conf={conf})")
    if stats:
        fig, ax = plt.subplots(figsize=(5, 3.2))
        ks = list(stats); rate = [stats[k][1]/stats[k][2] for k in ks]
        ax.bar(ks, rate, color=["#2980b9", "#e67e22"][:len(ks)])
        ax.set_ylabel("Fraccion de imagenes con >=1 FP"); ax.set_ylim(0, 1)
        ax.set_title(f"Falsos positivos sobre fondo puro (conf={conf})")
        for i, r in enumerate(rate): ax.text(i, r+0.02, f"{r:.2f}", ha="center")
        plt.tight_layout(); fig.savefig(OUT_DIR / "background_fp.png", bbox_inches="tight")
        print("Guardado:", OUT_DIR / "background_fp.png"); plt.show()

background_fp_probe()


## 13 · Galería de fallos (FP / misses) con CAM

Imágenes donde el modelo falla, con su EigenCAM P3, para ver los errores característicos (dónde miraba al
equivocarse). El criterio es de *presencia de clases* (GT vs predicción), no de localización.

In [ ]:
def failure_gallery(key, conf=0.25, max_show=6, imgsz=None):
    imgsz = imgsz or IMGSZ
    if key not in models or not YOLO_DATA.get(key): print(f"[{key}] sin YOLO_DATA."); return
    model = models[key]
    images_dir = YOLO_DATA[key]["images"]; labels_dir = YOLO_DATA[key]["labels"]
    if not Path(images_dir).exists(): print(f"[{key}] no existe {images_dir}."); return
    shown = 0
    for img_path in sorted(glob.glob(os.path.join(images_dir, "*")))[:400]:
        lab = Path(labels_dir) / (Path(img_path).stem + ".txt")
        gt = set()
        if lab.exists():
            for line in lab.read_text().strip().splitlines():
                if line.split(): gt.add(int(line.split()[0]))
        res = model.predict(img_path, imgsz=imgsz, conf=conf, verbose=False)[0]
        pred = set(int(c) for c in res.boxes.cls.tolist()) if res.boxes is not None else set()
        if gt == pred: continue
        cam, lb = eigencam(model, img_path, TARGET_LAYERS["P3"], imgsz)
        fig, ax = plt.subplots(1, 2, figsize=(9, 4))
        ax[0].imshow(res.plot()[:, :, ::-1]); clean_ax(ax[0])
        gtn = {model.names[i] for i in gt}; pn = {model.names[i] for i in pred}
        ax[0].set_title(f"GT={gtn}  Pred={pn}", fontsize=9)
        ax[1].imshow(overlay_cam(lb, cam)); ax[1].set_title("EigenCAM P3"); clean_ax(ax[1])
        plt.tight_layout()
        fig.savefig(OUT_DIR / f"galeria_fallo_{key}_{shown+1}.png", bbox_inches="tight")
        plt.show()
        shown += 1
        if shown >= max_show: break
    if shown == 0: print("No se encontraron discrepancias en el subconjunto.")
    else: print(f"Guardadas {shown} figuras: galeria_fallo_{key}_*.png")

# Para AMBOS modelos. Los fallos del unificado (drone->airplane) son muy ilustrativos de PI1.
for key, m in analyze_models():
    failure_gallery(key)

## 14 · Coste de la cuantización float → INT8

Validamos el `.pt` (float) y el `.tflite` (INT8) sobre el mismo conjunto y comparamos mAP. Justifica el
despliegue: *"INT8 cuesta X% de mAP a cambio de Y× de velocidad."*

> **Importante:** el TFLite tiene tamaño de entrada FIJO. Usa `imgsz` igual al del fichero (el `unified_int8.tflite`
> y `best_int8.tflite` son de **320**). Para el de 512 pasa `imgsz=512`. La validación TFLite en Colab CPU es lenta.

In [ ]:
def quant_gap(key, imgsz=320):
    if key not in models or not DATA_YAML.get(key) or not Path(DATA_YAML[key]).exists():
        print(f"[{key}] falta data.yaml."); return
    if not TFLITE.get(key) or not Path(TFLITE[key]).exists():
        print(f"[{key}] falta el TFLite INT8."); return
    yml = DATA_YAML[key]; split = pick_split(yml)
    fm = models[key].val(data=yml, imgsz=imgsz, split=split, verbose=False)
    im = YOLO(TFLITE[key]).val(data=yml, imgsz=imgsz, split=split, verbose=False)
    rows = [("mAP50", fm.box.map50, im.box.map50), ("mAP50-95", fm.box.map, im.box.map)]
    print(f"\n=== {key} — float vs INT8 @ imgsz={imgsz} (split={split}) ===")
    print(f"{'métrica':10s} {'float':>8s} {'INT8':>8s} {'Δ':>8s} {'Δ%':>7s}")
    labels, deltas = [], []
    for name, f, q in rows:
        d = q - f; dp = 100*d/(f+1e-9)
        print(f"{name:10s} {f:8.3f} {q:8.3f} {d:8.3f} {dp:6.1f}%")
        labels.append(name); deltas.append(dp)
    plt.figure(figsize=(5, 3.2))
    plt.bar(labels, deltas, color=["#c0392b" if d < 0 else "#27ae60" for d in deltas])
    plt.axhline(0, color="k", lw=0.8); plt.ylabel("Δ INT8 vs float (%)")
    plt.title(f"Coste de cuantización — {key} @ {imgsz}"); plt.tight_layout()
    plt.savefig(OUT_DIR / f"cuantizacion_{key}_{imgsz}.png", bbox_inches="tight"); plt.show()

TFLITE["dronebird"] = "/content/drive/MyDrive/TFG/models/tfg_drone_bird/weights/best_saved_model/best_full_integer_quant.tflite"
quant_gap("dronebird", imgsz=320)

# NO se ejecuta quant_gap('unified'): su TFLite desplegado es FP16 (el INT8 completo colapsa la
# cabeza de clasificacion, max score de clase ~0). Compararia float vs FP16 y FALSEARIA el coste
# de INT8. La AUSENCIA de esta figura para el unificado es deliberada, no un olvido.
print("\n[NOTA] Unificado -> desplegado en FP16. INT8 no viable (cabeza de clase colapsa).")
print("       Por eso NO hay figura de cuantizacion del unificado (documentado en la memoria).")

## 14b · Brecha de dominio: lo que ve el móvil ≠ lo que se entrenó

El tutor preguntó por la distorsión de cámara y por el efecto de que la entrada real difiera del dataset.
La distorsión geométrica del sensor principal del S22 es pequeña (óptica rectilínea), pero **sí** hay una
brecha de dominio: el móvil aplica realce (contraste 1.25, brillo +15) y **letterbox** antes de inferir,
sobre un sensor y una compresión distintos a los del dataset. Esta celda hace visible esa transformación
(dataset crudo → tubería de despliegue). La evidencia empírica del efecto ya está en el estudio de campo:
en test el unificado detecta el dron casi como el especialista, pero en campo lo etiqueta *airplane*
(desacople test↔campo = *domain shift*).


In [ ]:
def enhance_like_device(bgr, contrast=1.25, brightness=15):
    """Replica (aprox.) el enhanceBitmap de InferenceChannel.kt: contraste + brillo."""
    out = bgr.astype(np.float32) * contrast + brightness
    return np.clip(out, 0, 255).astype(np.uint8)

if Path(SAMPLE_IMG).exists():
    raw = cv2.imread(SAMPLE_IMG)
    lb_plain = letterbox(raw, IMGSZ)
    dev = letterbox(enhance_like_device(raw), IMGSZ)     # realce + letterbox = lo que ve el modelo
    fig, ax = plt.subplots(1, 3, figsize=(14, 4.6))
    ax[0].imshow(raw[:, :, ::-1]); ax[0].set_title("Dataset crudo"); clean_ax(ax[0])
    ax[1].imshow(lb_plain[:, :, ::-1]); ax[1].set_title(f"Letterbox {IMGSZ} (sin realce)"); clean_ax(ax[1])
    ax[2].imshow(dev[:, :, ::-1]); ax[2].set_title("Despliegue (realce + letterbox)"); clean_ax(ax[2])
    fig.suptitle("Brecha de dominio: transformacion de entrada del movil", fontsize=13)
    plt.tight_layout()
    fig.savefig(OUT_DIR / "domain_gap_demo.png", bbox_inches="tight")
    print("Guardado:", OUT_DIR / "domain_gap_demo.png"); plt.show()
else:
    print("Configura SAMPLE_IMG para la demo de brecha de dominio.")


## Cierre — mapa a la memoria

Hilo conductor = tus **dos afirmaciones centrales**:

- **Especialista vs generalista** → §11 (t-SNE, en especial la *comparación con los mismos crops*) + §7/§12
  (Grad-CAM por clase y confusión bird↔drone). Ahí está el *porqué*.
- **Degradación con la distancia** → §6 (EigenCAM P3 a distancias crecientes) junto al barrido de campo.

El resto sostiene el argumento: §5 (dónde mira), §8 (saliency), §9–10 (qué aprende), §13 (fallos), §14
(coste INT8 que justifica el despliegue).

**Orden sugerido:** §1b (chequeo) → 6 → 11 → 12 → 9/10 → 13 → 14 → 7/8. Empieza por el especialista
(`best_dronebird.pt`); repite §6, §11 y §12 con el unificado.

---
### Figuras nuevas de esta revisión (para cablear en la memoria)
Ambos modelos, sufijo `_unified` en el generalista:
`eigencam_multiescala[_unified]`, `barrido_distancia_P3[_unified]` (+`_letterbox`),
`gradcam_bird_drone` / `gradcam_unified`, `smoothgrad[_unified]`,
`saliency_vanilla_{dronebird,unified}`, `filtros_capa1[_unified]`,
`featuremaps_{temprana,P3,P5}[_unified]`, `confusion_{modelo}_{320,640}_counts` (background),
`roc_{dronebird,unified}`, `background_fp` (si hay `BACKGROUND_DIR`), `domain_gap_demo`.

**Mapa a la reunión del tutor:** (1) distorsión/brecha de dominio → §14b + campo; (2) ROC/AUC → §12b;
(3) background en confusión → §12/§12c; (4) Grad-CAM/saliency del unificado → §7/§8/§8b; (6) barrido
cuadrado → §6; (7) todo ×2 modelos → en cada sección.


## 15 · Descargar las figuras

Comprime `OUT_DIR` (las figuras que consume la memoria) y la descarga. Sube el contenido a la carpeta
`figuras/` de tu proyecto de Overleaf, junto a las figuras del estudio de campo.


In [ ]:
import shutil
from google.colab import files
zip_path = "/content/figuras_tfg"
shutil.make_archive(zip_path, "zip", OUT_DIR)
print("Figuras en:", OUT_DIR)
print("ZIP:", zip_path + ".zip")
files.download(zip_path + ".zip")
